In [3]:
import pandas as pd
import json
import re

# 1. JSONL 불러오기
records = []
with open(r'C:\workspace\finalproject\data\All_Beauty.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        records.append({
            'product_id': data.get('asin'),
            'rating': data.get('rating'),
            'review_text': data.get('text')
        })

df_json = pd.DataFrame(records)
print(f"총 행 수: {len(df_json)}")
print(df_json.head(3))

c:\Users\asiae\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\asiae\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


총 행 수: 701528
   product_id  rating                                        review_text
0  B00YQ6X8EO     5.0  This spray is really nice. It smells really go...
1  B081TJ8YS3     4.0  This product does what I need it to do, I just...
2  B07PNNCSP9     5.0                          Smells good, feels great!


In [4]:
# 2. 결측치 확인
print(df_json[['review_text', 'rating']].isnull().sum())
print(f"중복 행 수: {df_json.duplicated().sum()}")
print(df_json['rating'].value_counts().sort_index())

review_text    0
rating         0
dtype: int64
중복 행 수: 8505
rating
1.0    102080
2.0     43034
3.0     56307
4.0     79381
5.0    420726
Name: count, dtype: int64


In [5]:
# 전체 컬럼 기준 중복
print(f"전체 컬럼 중복: {df_json.duplicated().sum()}")

# review_text만 기준 중복
print(f"리뷰 텍스트 기준 중복: {df_json.duplicated(subset='review_text').sum()}")

전체 컬럼 중복: 8505
리뷰 텍스트 기준 중복: 57899


In [6]:
# 중복 제거 (리뷰 텍스트 기준)
df_json = df_json.drop_duplicates(subset='review_text')

# 3점 제거
df_json = df_json[df_json['rating'] != 3.0]

# 레이블 생성
df_json['label'] = df_json['rating'].apply(lambda x: 1 if x >= 4 else 0)

# 텍스트 정제
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_json['clean_review'] = df_json['review_text'].apply(clean_text)

# 빈 텍스트 제거
df_json = df_json[df_json['clean_review'].str.len() > 0]

# 한국어 제거
df_json = df_json[~df_json['review_text'].str.contains('[가-힣]', regex=True, na=False)]

# 최종 확인
print(f"최종 행 수: {len(df_json)}")
print(df_json['label'].value_counts())

최종 행 수: 589308
label
1    450184
0    139124
Name: count, dtype: int64


In [7]:
# 저장
df_json.to_csv(r'C:\workspace\finalproject\data\beauty_reviews_cleaned.csv', index=False)
print("저장 완료!")

저장 완료!


In [8]:
records = []
with open(r'C:\workspace\finalproject\data\meta_All_Beauty.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        records.append({
            'product_id': data.get('parent_asin'),
            'product_name': data.get('title'),
            'category': data.get('main_category'),
            'average_rating': data.get('average_rating'),
            'price': data.get('price'),
            'store': data.get('store')
        })

df_meta = pd.DataFrame(records)
print(f"총 행 수: {len(df_meta)}")
print(df_meta.head(3).to_string())

총 행 수: 112590
   product_id                                                                                                                                     product_name    category  average_rating  price                   store
0  B01CUPMQZE                                                                                              Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)  All Beauty             4.8    NaN         Howard Products
1  B076WQZGPM  Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.  All Beauty             4.5    NaN                  Yes To
2  B000B658RI                                                                                                 Eye Patch Black Adult with Tie Band (6 Per Pack)  All Beauty             4.4    NaN  Levine Health Products


In [9]:
print(f"price 결측치: {df_meta['price'].isnull().sum()}")
print(f"product_id 중복: {df_meta['product_id'].duplicated().sum()}")

price 결측치: 94886
product_id 중복: 0


In [10]:
# price 제거
df_meta = df_meta.drop(columns=['price'])

# 리뷰 데이터랑 merge
df_beauty = pd.read_csv(r'C:\workspace\finalproject\data\beauty_reviews_cleaned.csv')
df_merged = df_beauty.merge(df_meta, on='product_id', how='left')

# 확인
print(f"merge 후 행 수: {len(df_merged)}")
print(f"결측치:\n{df_merged.isnull().sum()}")
print(df_merged.head(3).to_string())

merge 후 행 수: 589308
결측치:
product_id            0
rating                0
review_text           4
label                 0
clean_review          0
product_name      51708
category          51708
average_rating    51708
store             91331
dtype: int64
   product_id  rating                                                                                                                                                                                                                                                                                                   review_text  label                                                                                                                                                                                                                                                                                        clean_review                                                                                                                            

In [11]:
# store 컬럼 제거
df_merged = df_merged.drop(columns=['store'])

# product_name, category, average_rating 결측치 행 제거
df_merged = df_merged.dropna(subset=['product_name', 'category', 'average_rating'])

# review_text 결측치 제거
df_merged = df_merged.dropna(subset=['review_text'])

# 최종 확인
print(f"최종 행 수: {len(df_merged)}")
print(f"결측치:\n{df_merged.isnull().sum()}")
print(df_merged['label'].value_counts())

최종 행 수: 537596
결측치:
product_id        0
rating            0
review_text       0
label             0
clean_review      0
product_name      0
category          0
average_rating    0
dtype: int64
label
1    408497
0    129099
Name: count, dtype: int64


In [12]:
df_merged.to_csv(r'C:\workspace\finalproject\data\beauty_final_cleaned.csv', index=False)
print("저장 완료!")

저장 완료!


In [13]:
import pandas as pd
import json

# 메타데이터 다시 로드
records = []
with open(r'C:\workspace\finalproject\data\meta_All_Beauty.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        records.append({
            'product_id': data.get('parent_asin'),
            'product_name': data.get('title'),
            'category': data.get('main_category'),
            'average_rating': data.get('average_rating'),
            'store': data.get('store'),
            'details': data.get('details')
        })

df_meta = pd.DataFrame(records)

# details 컬럼 샘플 확인
print(df_meta['details'].dropna().head(10).tolist())

[{'Package Dimensions': '7.1 x 5.5 x 3 inches; 2.38 Pounds', 'UPC': '617390882781'}, {'Item Form': 'Powder', 'Skin Type': 'Acne Prone', 'Brand': 'Yes To', 'Age Range (Description)': 'Adult', 'Unit Count': '10 Fl Oz', 'Is Discontinued By Manufacturer': 'No', 'Item model number': 'SG_B076WQZGPM_US', 'UPC': '653801351125', 'Manufacturer': 'Yes to Tomatoes'}, {'Manufacturer': 'Levine Health Products'}, {'Brand': 'Cherioll', 'Item Form': 'Powder', 'Finish Type': 'Natural', 'Product Benefits': 'Long Lasting', 'Skin Type': 'All', 'Package Dimensions': '8.43 x 5.91 x 0.87 inches; 8.78 Ounces', 'Item model number': 'eyebrow sticker001'}, {'UPC': '644287689178'}, {'Color': 'As Shown', 'Size': 'Large', 'Material': 'Acrylonitrile Butadiene Styrene (ABS)', 'Brand': 'Lurrose', 'Style': 'French', 'Product Dimensions': '5.63 x 2.83 x 0.39 inches; 1.9 Ounces', 'UPC': '799768026253', 'Manufacturer': 'Lurrose'}, {'Brand': 'Edoneery', 'Material': 'Silk', 'Number of Items': '1', 'Size': '1 Count (Pack of 1

In [14]:
# 전체 구조 파악
print(f"총 행 수: {len(df_meta)}")
print(f"\n컬럼별 결측치:")
print(df_meta.isnull().sum())
print(f"\n카테고리 종류:")
print(df_meta['category'].value_counts())
print(f"\ndetails 키 종류 파악:")
keys = {}
for d in df_meta['details'].dropna():
    for k in d.keys():
        keys[k] = keys.get(k, 0) + 1
print(sorted(keys.items(), key=lambda x: -x[1])[:20])

총 행 수: 112590

컬럼별 결측치:
product_id            0
product_name          0
category              0
average_rating        0
store             11331
details               0
dtype: int64

카테고리 종류:
category
All Beauty        112135
Premium Beauty       455
Name: count, dtype: int64

details 키 종류 파악:
[('Brand', 72113), ('Package Dimensions', 67825), ('UPC', 60252), ('Is Discontinued By Manufacturer', 39527), ('Item Form', 32439), ('Material', 30942), ('Hair Type', 26719), ('Unit Count', 23855), ('Product Dimensions', 23277), ('Age Range (Description)', 22929), ('Manufacturer', 19802), ('Item model number', 17951), ('Number of Items', 16964), ('Material Feature', 13153), ('Color', 11475), ('Skin Type', 11454), ('Style', 10667), ('Scent', 8428), ('Special Feature', 7653), ('Extension Length', 7254)]


In [15]:
# Item Form 종류 확인
item_forms = []
for d in df_meta['details'].dropna():
    if 'Item Form' in d:
        item_forms.append(d['Item Form'])

print(pd.Series(item_forms).value_counts().head(30))

Cream         4272
Liquid        3623
Gel           3386
Pair          1802
Powder        1783
Spray         1719
Oil           1649
Bar           1345
Lotion        1026
Pencil         885
Stick          791
Wand           705
Balm           680
Wrap           659
Scrunchie      598
Individual     526
Elastic        446
Sheet          371
Clay           351
Serum          337
Foam           310
Wax            232
Butter         199
Clip           192
Wipes          174
Spiral         173
Ribbon         138
spray          125
Mask           124
Aerosol        120
Name: count, dtype: int64


In [16]:
# product_name에서 카테고리 키워드 확인
keywords = {
    'skincare': ['moisturizer', 'serum', 'toner', 'cleanser', 'cream', 'lotion', 'essence', 'eye cream'],
    'suncare': ['sunscreen', 'spf', 'sun cream', 'uv', 'sunblock'],
    'mask': ['mask', 'sheet mask', 'face mask', 'peel off'],
    'makeup': ['foundation', 'lipstick', 'mascara', 'eyeshadow', 'blush', 'concealer', 'primer']
}

for cat, kws in keywords.items():
    count = df_meta['product_name'].str.lower().str.contains('|'.join(kws), na=False).sum()
    print(f"{cat}: {count}개")

skincare: 9265개
suncare: 3195개
mask: 2771개
makeup: 6811개


In [17]:
# 분류 안 된 것들 product_name 샘플 확인
unclassified = df_meta[
    ~df_meta['product_name'].str.lower().str.contains(
        'moisturizer|serum|toner|cleanser|cream|lotion|essence|eye cream|sunscreen|spf|sun cream|uv|sunblock|mask|sheet mask|face mask|peel off|foundation|lipstick|mascara|eyeshadow|blush|concealer|primer',
        na=False
    )
]
print(f"미분류 수: {len(unclassified)}")
print(unclassified['product_name'].head(30).tolist())

미분류 수: 92613
['Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)', 'Eye Patch Black Adult with Tie Band (6 Per Pack)', 'Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4D Imitation Eyebrow Tattoos, 4D Hair-like Authentic Eyebrows Waterproof Long Lasting for Woman & Man Makeup Tool', 'Precision Plunger Bars for Cartridge Grips – 93mm – Bag of 10 Plungers', 'Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY', 'Stain Bonnet For Baby Bonnet Silk Sleep Cap For Toddler Child Shower Cap Teens Kids', '50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic)', 'Gold extatic Musk EDT 90ml', '4 Pieces Satin Bonnet Adjustable Sleep Cap Double Layer Printed Hair Bonnet Large Reversible Sleeping Silk Night Hair Cap for Women Natural Curly Hair (Floral, Flamingo, Flower)', 'Stainless Steel Beard Comb, Portable Fashionable Engraving Design Mustache Hair Comb for Be

In [18]:
keywords = {
    'skincare': ['moisturizer', 'serum', 'toner', 'cleanser', 'cream', 'lotion', 
                 'essence', 'eye cream', 'exfoliant', 'retinol', 'niacinamide',
                 'hyaluronic', 'vitamin c', 'face wash', 'facial', 'anti-aging',
                 'brightening', 'acne', 'pore', 'hydrating', 'skin care'],
    'suncare': ['sunscreen', 'spf', 'sun cream', 'uv', 'sunblock', 'sun protection',
                'mineral sunscreen', 'zinc oxide', 'titanium dioxide'],
    'mask': ['mask', 'sheet mask', 'face mask', 'peel off', 'clay mask', 
             'mud mask', 'sleeping mask', 'lip mask'],
    'makeup': ['foundation', 'lipstick', 'mascara', 'eyeshadow', 'blush', 
               'concealer', 'primer', 'eyeliner', 'lip gloss', 'contour',
               'highlighter', 'bronzer', 'bb cream', 'cc cream', 'setting spray',
               'makeup', 'cosmetic', 'rouge', 'powder', 'lip liner', 'brow'],
    'haircare': ['shampoo', 'conditioner', 'hair mask', 'hair oil', 'hair serum',
                 'hair spray', 'hair color', 'hair dye', 'hair treatment',
                 'hair growth', 'scalp', 'argan', 'keratin', 'hair extensions',
                 'wig', 'braiding', 'hair', 'curl', 'frizz'],
    'fragrance': ['perfume', 'cologne', 'eau de', 'edt', 'edp', 'fragrance', 
                  'musk', 'scent', 'body spray', 'deodorant'],
    'nail': ['nail', 'manicure', 'pedicure', 'gel polish', 'nail art'],
    'tools': ['brush', 'sponge', 'applicator', 'comb', 'scissors', 'curler',
              'straightener', 'trimmer', 'razor', 'tweezer', 'roller'],
    'bodycare': ['body wash', 'body lotion', 'body cream', 'body scrub', 
                 'bath', 'soap', 'hand cream', 'foot cream', 'body butter'],
}

# 카테고리 분류 함수
def classify_product(name):
    name = str(name).lower()
    for cat, kws in keywords.items():
        if any(kw in name for kw in kws):
            return cat
    return 'other'

df_meta['new_category'] = df_meta['product_name'].apply(classify_product)
print(df_meta['new_category'].value_counts())

new_category
haircare     35811
other        22211
makeup       18325
skincare     13480
nail          7446
fragrance     4465
tools         3614
bodycare      3019
suncare       2563
mask          1656
Name: count, dtype: int64


In [19]:
# other로 분류된 것들 샘플 확인
other = df_meta[df_meta['new_category'] == 'other']
print(other['product_name'].head(30).tolist())

['Eye Patch Black Adult with Tie Band (6 Per Pack)', 'Precision Plunger Bars for Cartridge Grips – 93mm – Bag of 10 Plungers', 'Stain Bonnet For Baby Bonnet Silk Sleep Cap For Toddler Child Shower Cap Teens Kids', '50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic)', 'Small MONEY CLIP Leather Wallet ID Bag Cash Holder Credit Card Cover Case Pouch', 'Nomade By Chloe', 'TOODOO 8 Sets Mermaid Face Gems Glitter Sticker Rhinestone Bindis Crystal Face Tattoo Forehead Decorations for Women Favors (Delicate Pattern)', 'Zydeco Chop Chop Cajun Seasoning Base, 8 Ounce Resealable Bags (Pack of Two - 1 Pound Total)', 'Suavecito X Breast Cancer Solutions - Original Hold Pomade', 'Zoella Beauty Tutti Fruity Foam Sweet Foam Shower Gel 250ml', 'Bulk Herbs: Clay, White Kaolin', 'SERGE LUTENS PARFUMS La Fille De Berlin, Mini.03 oz', 'Wholesale CASE of 10 - GOJO Supro Max Cherry Hand Cleaner-Supro Max

In [20]:
# 4개 카테고리만 남기기
target_categories = ['skincare', 'suncare', 'mask', 'makeup']
df_meta_filtered = df_meta[df_meta['new_category'].isin(target_categories)]

print(f"필터링 후 행 수: {len(df_meta_filtered)}")
print(df_meta_filtered['new_category'].value_counts())

필터링 후 행 수: 36024
new_category
makeup      18325
skincare    13480
suncare      2563
mask         1656
Name: count, dtype: int64


In [21]:
import json

count = 0
with open(r'C:\workspace\finalproject\data\meta_All_Beauty.jsonl', 'r') as f:
    for line in f:
        count += 1

print(f"총 행 수: {count}")

총 행 수: 112590


In [22]:
df_meta.head(10)

,product_id,product_name,category,average_rating,store,details,new_category
0,B01CUPMQZE,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",All Beauty,4.8,Howard Products,{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,haircare
1,B076WQZGPM,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,All Beauty,4.5,Yes To,"{'Item Form': 'Powder', 'Skin Type': 'Acne Pro...",skincare
2,B000B658RI,Eye Patch Black Adult with Tie Band (6 Per Pack),All Beauty,4.4,Levine Health Products,{'Manufacturer': 'Levine Health Products'},other
3,B088FKY3VD,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",All Beauty,3.1,Cherioll,"{'Brand': 'Cherioll', 'Item Form': 'Powder', '...",makeup
4,B07NGFDN6G,Precision Plunger Bars for Cartridge Grips – 9...,All Beauty,4.3,Precision,{'UPC': '644287689178'},other
5,B07G9GWFSM,Lurrose 100Pcs Full Cover Fake Toenails Artifi...,All Beauty,3.7,Lurrose,"{'Color': 'As Shown', 'Size': 'Large', 'Materi...",nail
6,B08XZ97HFY,Stain Bonnet For Baby Bonnet Silk Sleep Cap Fo...,All Beauty,4.1,Edoneery,"{'Brand': 'Edoneery', 'Material': 'Silk', 'Num...",other
7,B08DNQTTQK,50 Pieces False Eyelash Packaging Box Empty Ey...,All Beauty,3.8,Maitys,{'Package Dimensions': '14.49 x 11.26 x 2.36 i...,other
8,B01ERJEGS6,Gold extatic Musk EDT 90ml,All Beauty,3.7,Balmain,"{'Brand': 'Balmain', 'Item Form': 'Spray', 'It...",fragrance
9,B08P7LXKP7,4 Pieces Satin Bonnet Adjustable Sleep Cap Dou...,All Beauty,4.3,Geyoga,{'Package Dimensions': '12.49 x 9.97 x 1.46 in...,haircare


In [24]:
import json
import pandas as pd

records = []
with open(r'C:\workspace\finalproject\data\meta_Beauty_and_Personal_Care.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        data = json.loads(line)
        records.append(data)

df_care = pd.DataFrame(records)
print(df_care.columns.tolist())
print(df_care.head(3))

['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']
  main_category                                              title  \
0    All Beauty  Shiyeen 10 Colors Hair Chalk for Girls Gift, K...   
1    All Beauty  Ebbfurln Bob Wig Human Hair, 13x4 HD Lace Fron...   
2    All Beauty  Makeup brush cleaner and dryer electronic spin...   

   average_rating  rating_number  \
0             3.9             57   
1             4.1             60   
2             4.4              7   

                                            features description  price  \
0  [🌼[MEET YOUR HAIR COLOR NEEDS] Bright color, h...          []    NaN   
1  [Frontal Wigs Human Hair Material: 100% unproc...          []  45.65   
2                                                 []          []    NaN   

                                              images  \
0  [{'thumb': 'https://m.media-

In [26]:
count = 0
with open(r'C:\workspace\finalproject\data\meta_Beauty_and_Personal_Care.jsonl', 'r') as f:
    for line in f:
        count += 1

print(f"총 행 수: {count}")

총 행 수: 1028914


In [27]:
import json
import pandas as pd

records = []
with open(r'C:\workspace\finalproject\data\meta_Beauty_and_Personal_Care.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        records.append({
            'product_id': data.get('parent_asin'),
            'product_name': data.get('title'),
            'categories': data.get('categories'),
            'average_rating': data.get('average_rating'),
        })

df_care = pd.DataFrame(records)
print(f"총 행 수: {len(df_care)}")
print(df_care['categories'].head(5).tolist())

총 행 수: 1028914
[['Beauty & Personal Care', 'Hair Care', 'Hair Coloring Products', 'Hair Chalk'], ['Beauty & Personal Care', 'Hair Care', 'Hair Extensions, Wigs & Accessories', 'Wigs'], ['Beauty & Personal Care', 'Tools & Accessories', 'Makeup Brushes & Tools', 'Brush Cleaners'], ['Beauty & Personal Care', 'Hair Care', 'Hair Cutting Tools', 'Hair Clippers & Accessories', 'Clipper Combs & Guides'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens']]


In [28]:
# categories 샘플 확인
print(df_care['categories'].dropna().head(20).tolist())

[['Beauty & Personal Care', 'Hair Care', 'Hair Coloring Products', 'Hair Chalk'], ['Beauty & Personal Care', 'Hair Care', 'Hair Extensions, Wigs & Accessories', 'Wigs'], ['Beauty & Personal Care', 'Tools & Accessories', 'Makeup Brushes & Tools', 'Brush Cleaners'], ['Beauty & Personal Care', 'Hair Care', 'Hair Cutting Tools', 'Hair Clippers & Accessories', 'Clipper Combs & Guides'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens'], ['Beauty & Personal Care', 'Makeup', 'Eyes', 'Eyeshadow'], ['Beauty & Personal Care', 'Tools & Accessories', 'Makeup Brushes & Tools', 'Eye', 'Mascara Brushes'], ['Beauty & Personal Care', 'Personal Care', 'Bath & Bathing Accessories', 'Bathing Accessories', 'Shower Caps'], ['Beauty & Personal Care', 'Fragrance', "Women's", 'Eau de Parfum'], ['Beauty & Personal Care', 'Hair Care', 'Hair Extensions, Wigs & Accessories', 'Hair Extensions'], ['Beauty & Personal Care', 'Skin Care', 'Face', 'Treatments & Mas

In [29]:
# 카테고리 매핑 함수
def classify_category(categories):
    if not categories:
        return None
    
    cat_str = ' '.join(categories).lower()
    
    # 스킨케어
    if any(kw in cat_str for kw in [
        'toner', 'essence', 'serum', 'ampoule', 'cream', 'lotion', 
        'mist', 'face oil', 'moisturizer', 'skin care', 'skincare',
        'face', 'cleanser', 'face wash', 'retinol', 'treatments'
    ]):
        # 선케어 먼저 체크 (선케어가 스킨케어에 포함될 수 있어서)
        if any(kw in cat_str for kw in ['sunscreen', 'sun care', 'tanning', 'spf']):
            # 선케어 중분류
            if 'stick' in cat_str: return '선케어_선스틱'
            elif 'cushion' in cat_str: return '선케어_선쿠션'
            elif 'spray' in cat_str or 'patch' in cat_str: return '선케어_선스프레이/선패치'
            elif 'tanning' in cat_str or 'after sun' in cat_str: return '선케어_태닝/애프터선'
            else: return '선케어_선크림'
        
        # 마스크팩 체크
        if any(kw in cat_str for kw in ['mask', 'sheet mask', 'pad', 'patch', 'pore strip']):
            if 'sheet' in cat_str: return '마스크팩_시트팩'
            elif 'pad' in cat_str: return '마스크팩_패드'
            elif 'patch' in cat_str: return '마스크팩_패치'
            else: return '마스크팩_코팩'
        
        # 스킨케어 중분류
        if any(kw in cat_str for kw in ['toner', 'mist']): return '스킨케어_스킨/토너'
        elif any(kw in cat_str for kw in ['serum', 'essence', 'ampoule']): return '스킨케어_에센스/세럼/앰플'
        elif 'cream' in cat_str: return '스킨케어_크림'
        elif 'lotion' in cat_str: return '스킨케어_로션'
        elif any(kw in cat_str for kw in ['oil', 'mist']): return '스킨케어_미스트/오일'
        else: return '스킨케어_크림'
    
    # 선케어
    elif any(kw in cat_str for kw in ['sunscreen', 'sun care', 'tanning', 'spf']):
        if 'stick' in cat_str: return '선케어_선스틱'
        elif 'cushion' in cat_str: return '선케어_선쿠션'
        elif 'spray' in cat_str or 'patch' in cat_str: return '선케어_선스프레이/선패치'
        elif 'tanning' in cat_str or 'after sun' in cat_str: return '선케어_태닝/애프터선'
        else: return '선케어_선크림'
    
    # 마스크팩
    elif any(kw in cat_str for kw in ['mask', 'sheet mask', 'treatments & masks']):
        if 'sheet' in cat_str: return '마스크팩_시트팩'
        elif 'pad' in cat_str: return '마스크팩_패드'
        elif 'patch' in cat_str: return '마스크팩_패치'
        else: return '마스크팩_코팩'
    
    # 메이크업
    elif any(kw in cat_str for kw in ['makeup', 'foundation', 'lipstick', 'lip', 
                                        'eyeshadow', 'mascara', 'eyeliner', 'blush',
                                        'concealer', 'primer', 'bb cream', 'cc cream']):
        if any(kw in cat_str for kw in ['lip', 'lipstick', 'lip gloss', 'lip liner']): return '메이크업_립'
        elif any(kw in cat_str for kw in ['foundation', 'concealer', 'primer', 'bb', 'cc', 'face']): return '메이크업_베이스'
        elif any(kw in cat_str for kw in ['eye', 'eyeshadow', 'mascara', 'eyeliner']): return '메이크업_아이'
        else: return '메이크업_베이스'
    
    return None

df_care['new_category'] = df_care['categories'].apply(classify_category)

# 결과 확인
print(df_care['new_category'].value_counts())
print(f"\n분류 안 된 수: {df_care['new_category'].isnull().sum()}")

new_category
스킨케어_크림           183734
메이크업_베이스           65234
메이크업_립             64710
마스크팩_코팩            38824
메이크업_아이            37246
스킨케어_로션            17379
선케어_태닝/애프터선        12207
스킨케어_미스트/오일         7361
스킨케어_스킨/토너          4684
스킨케어_에센스/세럼/앰플      2336
마스크팩_패드              248
선케어_선크림                5
Name: count, dtype: int64

분류 안 된 수: 594946


In [30]:
# 분류 안 된 것들 categories 샘플 확인
unclassified = df_care[df_care['new_category'].isnull()]
print(f"미분류 수: {len(unclassified)}")
print(unclassified['categories'].head(20).tolist())

미분류 수: 594946
[['Beauty & Personal Care', 'Hair Care', 'Hair Coloring Products', 'Hair Chalk'], ['Beauty & Personal Care', 'Hair Care', 'Hair Extensions, Wigs & Accessories', 'Wigs'], ['Beauty & Personal Care', 'Personal Care', 'Bath & Bathing Accessories', 'Bathing Accessories', 'Shower Caps'], ['Beauty & Personal Care', 'Fragrance', "Women's", 'Eau de Parfum'], ['Beauty & Personal Care', 'Hair Care', 'Hair Extensions, Wigs & Accessories', 'Hair Extensions'], ['Beauty & Personal Care', 'Foot, Hand & Nail Care', 'Nail Art & Polish', 'Nail Art Accessories', 'Rhinestones'], ['Beauty & Personal Care', 'Hair Care', 'Hair Extensions, Wigs & Accessories', 'Wigs'], ['Beauty & Personal Care', 'Hair Care', 'Hair Extensions, Wigs & Accessories', 'Hair Extensions'], ['Beauty & Personal Care', 'Personal Care', 'Deodorants & Antiperspirants', 'Antiperspirant Deodorant'], ['Beauty & Personal Care', 'Hair Care', 'Hair Extensions, Wigs & Accessories', 'Hairpieces'], ['Beauty & Personal Care', 'Tools &

In [31]:
# 선케어 관련 카테고리 확인
suncare = df_care[df_care['categories'].apply(
    lambda x: any('sunscreen' in str(c).lower() or 'sun care' in str(c).lower() for c in x) if x else False
)]
print(f"선케어 관련 수: {len(suncare)}")
print(suncare['categories'].head(10).tolist())


선케어 관련 수: 12212
[['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Self-Tanners'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Self-Tanners', 'Body Self-Tanners'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Self-Tanners', 'Body Self-Tanners'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'After Sun'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens'], ['Beauty & Personal Care', 'Skin Care', 'Sunscree

In [32]:
def classify_category(categories):
    if not categories:
        return None
    
    cat_str = ' '.join(categories).lower()
    
    # 선케어 먼저 체크 (스킨케어보다 앞에!)
    if any(kw in cat_str for kw in ['sunscreen', 'tanning', 'self-tanner', 'after sun', 'spf']):
        if 'stick' in cat_str: return '선케어_선스틱'
        elif 'cushion' in cat_str: return '선케어_선쿠션'
        elif 'spray' in cat_str or 'patch' in cat_str: return '선케어_선스프레이/선패치'
        elif any(kw in cat_str for kw in ['tanning', 'self-tanner', 'after sun']): return '선케어_태닝/애프터선'
        else: return '선케어_선크림'
    
    # 마스크팩
    elif any(kw in cat_str for kw in ['mask', 'treatments & masks', 'peel']):
        if 'sheet' in cat_str: return '마스크팩_시트팩'
        elif 'pad' in cat_str: return '마스크팩_패드'
        elif 'patch' in cat_str: return '마스크팩_패치'
        else: return '마스크팩_코팩'
    
    # 스킨케어
    elif any(kw in cat_str for kw in ['skin care', 'toner', 'essence', 'serum', 
                                        'ampoule', 'cream', 'lotion', 'mist', 
                                        'moisturizer', 'cleanser', 'face wash', 'retinol']):
        if any(kw in cat_str for kw in ['toner', 'mist']): return '스킨케어_스킨/토너'
        elif any(kw in cat_str for kw in ['serum', 'essence', 'ampoule']): return '스킨케어_에센스/세럼/앰플'
        elif 'cream' in cat_str: return '스킨케어_크림'
        elif 'lotion' in cat_str: return '스킨케어_로션'
        elif any(kw in cat_str for kw in ['oil', 'mist']): return '스킨케어_미스트/오일'
        else: return '스킨케어_크림'
    
    # 메이크업
    elif any(kw in cat_str for kw in ['makeup', 'foundation', 'lipstick', 'lip',
                                        'eyeshadow', 'mascara', 'eyeliner', 'blush',
                                        'concealer', 'primer', 'bb cream', 'cc cream', 'eyes', 'lips', 'face']):
        if any(kw in cat_str for kw in ['lip', 'lipstick', 'lips']): return '메이크업_립'
        elif any(kw in cat_str for kw in ['eye', 'eyeshadow', 'mascara', 'eyeliner', 'eyes']): return '메이크업_아이'
        else: return '메이크업_베이스'
    
    return None

df_care['new_category'] = df_care['categories'].apply(classify_category)

print(df_care['new_category'].value_counts())
print(f"\n분류 안 된 수: {df_care['new_category'].isnull().sum()}")

new_category
스킨케어_크림           143300
메이크업_베이스           71655
메이크업_아이            67086
메이크업_립             64710
마스크팩_코팩            38824
스킨케어_로션            17379
선케어_태닝/애프터선        12207
스킨케어_미스트/오일         7361
스킨케어_스킨/토너          4684
스킨케어_에센스/세럼/앰플      2336
Name: count, dtype: int64

분류 안 된 수: 599372


In [34]:
# Sunscreens 카테고리 직접 확인
sunscreen = df_care[df_care['categories'].apply(
    lambda x: 'Sunscreens' in x if x else False
)]
print(f"Sunscreens 포함 수: {len(sunscreen)}")
print(sunscreen['categories'].head(5).tolist())
print(sunscreen['new_category'].value_counts())

Sunscreens 포함 수: 6812
[['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens'], ['Beauty & Personal Care', 'Skin Care', 'Sunscreens & Tanning Products', 'Sunscreens', 'Body Sunscreens']]
new_category
선케어_태닝/애프터선    6812
Name: count, dtype: int64


In [33]:
# 미분류 카테고리 전체 분포 확인
unclassified = df_care[df_care['new_category'].isnull()]

# 두번째 카테고리(대분류) 기준으로 분포 확인
second_cat = unclassified['categories'].apply(lambda x: x[1] if x and len(x) > 1 else None)
print(second_cat.value_counts())

categories
Hair Care                 274409
Foot, Hand & Nail Care    116394
Tools & Accessories        57163
Fragrance                  52427
Personal Care              42988
                           ...  
elf                            1
Karuna                         1
Emerald Tones                  1
2022 BFCM Preview              1
Beauty Sales & Deals           1
Name: count, Length: 108, dtype: int64


In [35]:
# 두번째 카테고리(대분류) 전체 분포 확인
second_cat = df_care['categories'].apply(lambda x: x[1] if x and len(x) > 1 else None)
print(second_cat.value_counts())

categories
Hair Care                 307761
Skin Care                 199381
Foot, Hand & Nail Care    128087
Makeup                    120661
Tools & Accessories       114709
                           ...  
elf                            1
Karuna                         1
Emerald Tones                  1
2022 BFCM Preview              1
Beauty Sales & Deals           1
Name: count, Length: 118, dtype: int64


In [36]:
df_care['categories'].head(10).tolist()

[['Beauty & Personal Care',
  'Hair Care',
  'Hair Coloring Products',
  'Hair Chalk'],
 ['Beauty & Personal Care',
  'Hair Care',
  'Hair Extensions, Wigs & Accessories',
  'Wigs'],
 ['Beauty & Personal Care',
  'Tools & Accessories',
  'Makeup Brushes & Tools',
  'Brush Cleaners'],
 ['Beauty & Personal Care',
  'Hair Care',
  'Hair Cutting Tools',
  'Hair Clippers & Accessories',
  'Clipper Combs & Guides'],
 ['Beauty & Personal Care',
  'Skin Care',
  'Sunscreens & Tanning Products',
  'Sunscreens',
  'Body Sunscreens'],
 ['Beauty & Personal Care', 'Makeup', 'Eyes', 'Eyeshadow'],
 ['Beauty & Personal Care',
  'Tools & Accessories',
  'Makeup Brushes & Tools',
  'Eye',
  'Mascara Brushes'],
 ['Beauty & Personal Care',
  'Personal Care',
  'Bath & Bathing Accessories',
  'Bathing Accessories',
  'Shower Caps'],
 ['Beauty & Personal Care', 'Fragrance', "Women's", 'Eau de Parfum'],
 ['Beauty & Personal Care',
  'Hair Care',
  'Hair Extensions, Wigs & Accessories',
  'Hair Extensions']]

In [37]:
# 1~3번째 카테고리 확인
print(df_care['categories'].apply(lambda x: x[1] if x and len(x) > 1 else None).value_counts().head(20))
print()
print(df_care['categories'].apply(lambda x: x[2] if x and len(x) > 2 else None).value_counts().head(20))

categories
Hair Care                        307761
Skin Care                        199381
Foot, Hand & Nail Care           128087
Makeup                           120661
Tools & Accessories              114709
Fragrance                         52427
Shave & Hair Removal              44683
Personal Care                     42988
Salon & Spa Equipment              7880
Salon & Spa                        5575
National Lipstick Day              2516
Men's Grooming                     1337
Wisp                                210
Gift Sets                           134
Beauty Outlet                        77
INGS_Consumables                     73
Indie Beauty                         53
Herbal Beauty & Personal Care        34
Beauty Markdowns                     27
Beauty $5 Store                      21
Name: count, dtype: int64

categories
Face                                   109351
Hair Extensions, Wigs & Accessories    100933
Nail Art & Polish                       95846
Body         

In [38]:
# Skin Care 중분류
print("=== Skin Care 중분류 ===")
skincare_mid = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Skin Care' if x and len(x) > 1 else False
)]['categories'].apply(lambda x: x[2] if len(x) > 2 else None)
print(skincare_mid.value_counts())

print()

# Makeup 중분류
print("=== Makeup 중분류 ===")
makeup_mid = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Makeup' if x and len(x) > 1 else False
)]['categories'].apply(lambda x: x[2] if len(x) > 2 else None)
print(makeup_mid.value_counts())

=== Skin Care 중분류 ===
categories
Body                             82677
Face                             78215
Sunscreens & Tanning Products    12206
Lip Care                         10904
Eyes                             10023
Sets & Kits                       3497
Maternity                          411
Name: count, dtype: int64

=== Makeup 중분류 ===
categories
Eyes               39162
Face               31135
Lips               29783
Body               12116
Makeup Palettes     2971
Makeup Sets         2544
Makeup Remover      1944
Name: count, dtype: int64


In [39]:
# Makeup > Body 샘플 확인
makeup_body = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Makeup' and len(x) > 2 and x[2] == 'Body' if x and len(x) > 2 else False
)]
print(makeup_body['product_name'].head(20).tolist())

['16Pcs Halloween Eyeshadow Eyeliner Sticker Spider Web Skull Bat Eye Shadow Decals for Women Halloween Masquerade Party Face Eye Realistic Makeup 3D Self- Adhesive Eye Art Decoration Tools', 'Kyerivs Face Jewels Stickers Women Crystal Rhinestone Multicolor Crystals Sticker Temporary Stickers Forehead Body Decorations& Freebie Inside (Pattern Set 1)', 'SIQUK 15 Sets Face Gems Glitter Mermaid Face Jewels Crystal Stickers with 15 Boxes Chunky Face Glitter for Festival Rave Carnival Party', 'BenRan 60 Pcs Day of the Dead Face Tattoos, (2022 NEW)Skull Halloween Temporary Face Tattoo, Día de Los Muertos Skeleton Fake Face Tattoos Makeup for Men Women Kids Halloween Makeup', 'Face Painting Makeup – ProAiir Water Resistant Makeup - 2.1 oz (60ml) Bubblegum Pink', 'Bachelorettesy Tropical Bachelorette Tattoos', 'Nickelodeon Teenage Mutant Ninja Turtles 75 Tattoos', 'Casciybo Flower Temporary Tattoos for Kids Girl, 10 Sheets Fake Waterproof Cute Small Tattoo Stickers for Child Birthday Party Fav

In [40]:
# Skin Care > Face 소분류 확인
face_sub = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Skin Care' and len(x) > 2 and x[2] == 'Face' if x and len(x) > 2 else False
)]['categories'].apply(lambda x: x[3] if len(x) > 3 else None)
print(face_sub.value_counts())

categories
Treatments & Masks       30858
Creams & Moisturizers    21706
Cleansers                15358
Toners & Astringents      3806
Sets & Kits               3134
Polishes & Scrubs         2186
Name: count, dtype: int64


In [41]:
# Skin Care > Face > Treatments & Masks 소분류 확인
mask_sub = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Skin Care' and len(x) > 3 and x[2] == 'Face' and x[3] == 'Treatments & Masks' if x and len(x) > 3 else False
)]['categories'].apply(lambda x: x[4] if len(x) > 4 else None)
print(mask_sub.value_counts())

categories
Masks                    13836
Serums                   10951
Facial Peels              1860
Pore Cleansing Strips      784
Microdermabrasion          486
Name: count, dtype: int64


In [43]:
# 사용할 카테고리 필터링
skincare_include = ['Body', 'Face', 'Sunscreens & Tanning Products', 'Lip Care', 'Eyes']
makeup_include = ['Eyes', 'Face', 'Lips', 'Makeup Palettes', 'Makeup Sets']

def classify_category(categories):
    if not categories or len(categories) < 3:
        return None
    
    level1 = categories[1]  # Skin Care / Makeup
    level2 = categories[2]  # 중분류
    
    if level1 == 'Skin Care' and level2 in skincare_include:
        if level2 == 'Sunscreens & Tanning Products':
            return 'suncare'
        elif level2 == 'Lip Care':
            return 'skincare_lipcare'
        elif level2 == 'Eyes':
            return 'skincare_eyecare'
        else:
            return f'skincare_{level2}'
    
    elif level1 == 'Makeup' and level2 in makeup_include:
        return f'makeup_{level2}'
    
    return None

df_care['new_category'] = df_care['categories'].apply(classify_category)

print(df_care['new_category'].value_counts())
print(f"\n분류 안 된 수: {df_care['new_category'].isnull().sum()}")

new_category
skincare_Body             82677
skincare_Face             78215
makeup_Eyes               39162
makeup_Face               31135
makeup_Lips               29783
suncare                   12206
skincare_lipcare          10904
skincare_eyecare          10023
makeup_Makeup Palettes     2971
makeup_Makeup Sets         2544
Name: count, dtype: int64

분류 안 된 수: 729294


In [44]:
def classify_category(categories):
    if not categories or len(categories) < 3:
        return None
    
    level1 = categories[1]
    level2 = categories[2] if len(categories) > 2 else None
    level3 = categories[3] if len(categories) > 3 else None
    level4 = categories[4] if len(categories) > 4 else None

    if level1 == 'Skin Care':
        if level2 == 'Sunscreens & Tanning Products':
            return 'suncare'
        elif level2 == 'Face':
            if level3 == 'Treatments & Masks':
                if level4 == 'Masks':
                    return 'maskpack'
                else:
                    return 'skincare_Face'
            else:
                return 'skincare_Face'
        elif level2 == 'Body': return 'skincare_Body'
        elif level2 == 'Lip Care': return 'skincare_lipcare'
        elif level2 == 'Eyes': return 'skincare_eyecare'
    
    elif level1 == 'Makeup':
        if level2 == 'Eyes': return 'makeup_Eyes'
        elif level2 == 'Face': return 'makeup_Face'
        elif level2 == 'Lips': return 'makeup_Lips'
        elif level2 == 'Makeup Palettes': return 'makeup_Makeup Palettes'
        elif level2 == 'Makeup Sets': return 'makeup_Makeup Sets'
    
    return None

df_care['new_category'] = df_care['categories'].apply(classify_category)

print(df_care['new_category'].value_counts())
print(f"\n분류 안 된 수: {df_care['new_category'].isnull().sum()}")

new_category
skincare_Body             82677
skincare_Face             64379
makeup_Eyes               39162
makeup_Face               31135
makeup_Lips               29783
maskpack                  13836
suncare                   12206
skincare_lipcare          10904
skincare_eyecare          10023
makeup_Makeup Palettes     2971
makeup_Makeup Sets         2544
Name: count, dtype: int64

분류 안 된 수: 729294


# 각 카테고리별 상세 분류 봐보기 
+ ex) skincare_body > 소분류 


In [45]:
# skincare_Body 소분류 확인
body_detail = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Skin Care' and len(x) > 2 and x[2] == 'Body' if x and len(x) > 2 else False
)]['categories'].apply(lambda x: x[3] if len(x) > 3 else None)
print("=== skincare_Body 소분류 ===")
print(body_detail.value_counts())

=== skincare_Body 소분류 ===
categories
Cleansers       38025
Moisturizers    35316
Body Scrubs      5550
Sets & Kits      3212
Body Mud          203
Name: count, dtype: int64


In [46]:
# skincare_Face 소분류 확인
face_detail = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Skin Care' and len(x) > 2 and x[2] == 'Face' if x and len(x) > 2 else False
)]['categories'].apply(lambda x: x[3] if len(x) > 3 else None)
print("=== skincare_Face 소분류 ===")
print(face_detail.value_counts())

print()

=== skincare_Face 소분류 ===
categories
Treatments & Masks       30858
Creams & Moisturizers    21706
Cleansers                15358
Toners & Astringents      3806
Sets & Kits               3134
Polishes & Scrubs         2186
Name: count, dtype: int64



In [47]:
# Cleansers 샘플 확인
cleansers = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Skin Care' and len(x) > 3 and x[2] == 'Face' and x[3] == 'Cleansers' if x and len(x) > 3 else False
)]
print("=== Cleansers 샘플 ===")
print(cleansers['product_name'].head(10).tolist())

print()

# Sets & Kits 샘플 확인
sets = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Skin Care' and len(x) > 3 and x[2] == 'Face' and x[3] == 'Sets & Kits' if x and len(x) > 3 else False
)]
print("=== Sets & Kits 샘플 ===")
print(sets['product_name'].head(10).tolist())

=== Cleansers 샘플 ===
['Holika Holika Pig Nose Clear Black Head Cleansing Sugar Scrub, 1.01 Ounce', 'OM Botanical One Step Face Cleanser, All-in-One Vegan Exfoliating Facial Cleanser and Toner for Blemished-Free Skin, Natural Aloe-Based Face Wash for Women and Men, Non-Drying Face Exfoliator and pH-Balanced, 100mL', 'chhwau Form Whip Maker，Face Wash Foamer，Rich Foam Maker for Face Wash，Foam Cleanser and Face Wash ，Skincare Tools (blue)', 'Santa Marche 200g Hot Gel Cleansing Orange Make Last Joke From Japan', 'HARVANCO Double-sided Two-tone Reusable Makeup Remover Cloths - Washable Microfiber Makeup Removal Cloth - Upgraded Executive Makeup Remover and Facial Cleansing Cloth, 6 pack（8.6 x 8.6 in & 6 x 6 in, Pink-White）', 'Natural Facial Cleanser with Vitamin C - Natural Anti Oxidant & Anti-Aging Face Wash for Women - Loaded with Powerful Antioxidants: Green Tea & Rose Hip Oil for Fine Lines & Wrinkles', 'JewelryWe 3pcs Reusable Makeup Remover Pads Double-Side Washable Makeup Removal Clot

In [52]:
import pandas as pd
pd.set_option('display.max_rows', None)

# makeup Eyes 소분류
eyes_detail = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Makeup' and len(x) > 2 and x[2] == 'Eyes' if x and len(x) > 2 else False
)]['categories'].apply(lambda x: x[3] if len(x) > 3 else None)
print("=== makeup_Eyes 소분류 ===")
print(eyes_detail.value_counts())

print()

# makeup Face 소분류
face_detail = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Makeup' and len(x) > 2 and x[2] == 'Face' if x and len(x) > 2 else False
)]['categories'].apply(lambda x: x[3] if len(x) > 3 else None)
print("=== makeup_Face 소분류 ===")
print(face_detail.value_counts())

print()


=== makeup_Eyes 소분류 ===
categories
Eyeshadow                      12148
Eyeliner                        8732
Eyebrow Color                   7815
Mascara                         6428
Lash Enhancers & Primers        2032
Concealer                        257
Liner & Shadow Combinations      238
Eyeshadow Bases & Primers        236
Name: count, dtype: int64

=== makeup_Face 소분류 ===
categories
Foundation                   10964
Powder                        4723
Blush                         4171
Concealers & Neutralizers     3820
Foundation Primers            2123
Highlighters & Luminizers     2060
BB Creams                      842
Bronzers                       809
CC Creams                      520
Tinted Moisturizers            476
DD Creams                       48
Name: count, dtype: int64



In [51]:
# makeup Lips 소분류
lips_detail = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Makeup' and len(x) > 2 and x[2] == 'Lips' if x and len(x) > 2 else False
)]['categories'].apply(lambda x: x[3] if len(x) > 3 else None)
print("=== makeup_Lips 소분류 ===")
print(lips_detail.value_counts())

=== makeup_Lips 소분류 ===
categories
Lipstick            16407
Lip Glosses          9085
Lip Liners           2116
Lip Stains           1382
Lipstick Primers       70
Name: count, dtype: int64


In [53]:
# suncare 소분류
suncare_detail = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Skin Care' and len(x) > 2 and x[2] == 'Sunscreens & Tanning Products' if x and len(x) > 2 else False
)]['categories'].apply(lambda x: x[3] if len(x) > 3 else None)
print("=== suncare 소분류 ===")
print(suncare_detail.value_counts())

print()


=== suncare 소분류 ===
categories
Sunscreens                6812
Self-Tanners              3329
After Sun                  942
Tanning Oils & Lotions     912
Name: count, dtype: int64



In [55]:
# skincare_lipcare 소분류
lipcare_detail = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Skin Care' and len(x) > 2 and x[2] == 'Lip Care' if x and len(x) > 2 else False
)]['categories'].apply(lambda x: x[3] if len(x) > 3 else None)
print("=== skincare_lipcare 소분류 ===")
print(lipcare_detail.value_counts())

print()
# skincare_eyecare 소분류
eyecare_detail = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Skin Care' and len(x) > 2 and x[2] == 'Eyes' if x and len(x) > 2 else False
)]['categories'].apply(lambda x: x[3] if len(x) > 3 else None)
print("=== skincare_eyecare 소분류 ===")
print(eyecare_detail.value_counts())

print()


=== skincare_lipcare 소분류 ===
categories
Balms & Moisturizers    9492
Lip Plumpers             432
Scrubs                   385
Lip Butters              103
Name: count, dtype: int64

=== skincare_eyecare 소분류 ===
categories
Masks                     3868
Creams                    3061
Serums                     870
Gels                       752
Wrinkle Pads & Patches     248
Balms                       70
Name: count, dtype: int64



In [57]:
# makeup_Makeup Palettes 샘플
palettes = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Makeup' and len(x) > 2 and x[2] == 'Makeup Palettes' if x and len(x) > 2 else False
)]
print("=== makeup_Makeup Palettes 샘플 ===")
print(palettes['product_name'].head(5).tolist())

print()

# makeup_Makeup Sets 샘플
sets = df_care[df_care['categories'].apply(
    lambda x: x[1] == 'Makeup' and len(x) > 2 and x[2] == 'Makeup Sets' if x and len(x) > 2 else False
)]
print("=== makeup_Makeup Sets 샘플 ===")
print(sets['product_name'].head(5).tolist())


=== makeup_Makeup Palettes 샘플 ===
['Morphe Pro 35 Color Eyeshadow Makeup Palette - Its Bling (Highly Pigmented) 35E', 'e.l.f. Shimmer Palette, Shimmer Palette, 0.25 Ounce', 'Tarte Clay Play Face Shaping Palette', 'FLOWER BEAUTY CONTOUR PALETTE | Lift & Sculpt Contouring Palette | 3 Powder Makeup Shades to Sculpt, Blush & Highlight Face | Cruelty Free (Medium/Dark)', 'Ruby Kisses 3D Powder Contour Artist (Cream, Medium Dark) Define your cheekbones, perfect your nose, and sculpt your jawline']

=== makeup_Makeup Sets 샘플 ===
['FantasyDay All-in-one Holiday Make up Gift Set | Makeup Kit for Women Full Kit Essential Starter Bundle Include Eyeshadow Palette Lipstick Blush Foundation Concealer Face Powder Mascara Lipgloss Brush', 'Accforcity Purple New Ipad3 , Ipad2 Case with Keyboard & Bluetooth , Leather Case for Apple New Ipad3 , Ipad 2', 'Estee Lauder Limited Edition Pure Color Gold Great Makeup Gift SET', 'Makeup Set Kabuki Brush & Blush - By Jing Ai - Two Shades Light, Medium Or Dark Fo

In [59]:
def classify_category(categories):
    if not categories or len(categories) < 3:
        return None, None
    
    level1 = categories[1]
    level2 = categories[2] if len(categories) > 2 else None
    level3 = categories[3] if len(categories) > 3 else None
    level4 = categories[4] if len(categories) > 4 else None

    if level1 == 'Skin Care':
        if level2 == 'Sunscreens & Tanning Products':
            return 'suncare', level3
        elif level2 == 'Face':
            if level3 == 'Treatments & Masks':
                if level4 == 'Masks':
                    return 'maskpack', level4
                else:
                    return 'skincare_Face', level3
            elif level3 == 'Sets & Kits':
                return None, None
            else:
                return 'skincare_Face', level3
        elif level2 == 'Lip Care':
            return 'skincare_lipcare', level3
        elif level2 == 'Eyes':
            return 'skincare_eyecare', level3

    elif level1 == 'Makeup':
        if level2 == 'Eyes': return 'makeup_Eyes', level3
        elif level2 == 'Face': return 'makeup_Face', level3
        elif level2 == 'Lips': return 'makeup_Lips', level3
        elif level2 == 'Makeup Palettes': return 'makeup_Makeup Palettes', level3

    return None, None

df_care[['main_category', 'sub_category']] = df_care['categories'].apply(
    lambda x: pd.Series(classify_category(x))
)

# 필요한 카테고리만 필터링
df_filtered = df_care[df_care['main_category'].notna()].copy()

print(f"필터링 후 행 수: {len(df_filtered)}")
print(df_filtered['main_category'].value_counts())
print()
print(df_filtered[['product_id', 'main_category', 'sub_category', 'average_rating']].head(5).to_string())

필터링 후 행 수: 211265
main_category
skincare_Face             61245
makeup_Eyes               39162
makeup_Face               31135
makeup_Lips               29783
maskpack                  13836
suncare                   12206
skincare_lipcare          10904
skincare_eyecare          10023
makeup_Makeup Palettes     2971
Name: count, dtype: int64

    product_id  main_category        sub_category  average_rating
4   B00N4LMZZK        suncare          Sunscreens             4.0
5   B01DX1OEFO    makeup_Eyes           Eyeshadow             4.3
10  B01KQNJCNG  skincare_Face  Treatments & Masks             3.5
17  B07QX3LQQ2    makeup_Face          Foundation             3.4
18  B00N54K1UM    makeup_Lips            Lipstick             4.2


In [60]:
# 필요한 컬럼만 선택해서 저장
df_filtered[['product_id','main_category', 'sub_category', 'average_rating']].to_csv(
    r'C:\workspace\finalproject\data\meta_beauty_care_cleaned.csv', index=False
)
print("저장 완료!")
print(f"최종 행 수: {len(df_filtered)}")

저장 완료!
최종 행 수: 211265


In [61]:
import json
import pandas as pd

chunk = []
chunk_size = 100000  # 10만개씩
chunk_num = 0

with open(r'C:\workspace\finalproject\data\Beauty_and_Personal_Care.jsonl', 'r') as f:
    for i, line in enumerate(f):
        data = json.loads(line)
        chunk.append({
            'product_id': data.get('asin'),
            'rating': data.get('rating'),
            'review_text': data.get('text'),
            'timestamp': data.get('timestamp'),
            'verified_purchase': data.get('verified_purchase')
        })
        
        if len(chunk) == chunk_size:
            pd.DataFrame(chunk).to_parquet(
                rf'C:\workspace\finalproject\data\reviews_chunk_{chunk_num}.parquet',
                index=False
            )
            print(f"chunk_{chunk_num} 저장 완료 ({i+1}번째 줄)")
            chunk = []
            chunk_num += 1

# 남은 데이터 저장
if chunk:
    pd.DataFrame(chunk).to_parquet(
        rf'C:\workspace\finalproject\data\reviews_chunk_{chunk_num}.parquet',
        index=False
    )
    print(f"chunk_{chunk_num} 저장 완료")

print(f"\n총 {chunk_num + 1}개 파일 저장 완료!")

chunk_0 저장 완료 (100000번째 줄)
chunk_1 저장 완료 (200000번째 줄)
chunk_2 저장 완료 (300000번째 줄)
chunk_3 저장 완료 (400000번째 줄)
chunk_4 저장 완료 (500000번째 줄)
chunk_5 저장 완료 (600000번째 줄)
chunk_6 저장 완료 (700000번째 줄)
chunk_7 저장 완료 (800000번째 줄)
chunk_8 저장 완료 (900000번째 줄)
chunk_9 저장 완료 (1000000번째 줄)
chunk_10 저장 완료 (1100000번째 줄)
chunk_11 저장 완료 (1200000번째 줄)
chunk_12 저장 완료 (1300000번째 줄)
chunk_13 저장 완료 (1400000번째 줄)
chunk_14 저장 완료 (1500000번째 줄)
chunk_15 저장 완료 (1600000번째 줄)
chunk_16 저장 완료 (1700000번째 줄)
chunk_17 저장 완료 (1800000번째 줄)
chunk_18 저장 완료 (1900000번째 줄)
chunk_19 저장 완료 (2000000번째 줄)
chunk_20 저장 완료 (2100000번째 줄)
chunk_21 저장 완료 (2200000번째 줄)
chunk_22 저장 완료 (2300000번째 줄)
chunk_23 저장 완료 (2400000번째 줄)
chunk_24 저장 완료 (2500000번째 줄)
chunk_25 저장 완료 (2600000번째 줄)
chunk_26 저장 완료 (2700000번째 줄)
chunk_27 저장 완료 (2800000번째 줄)
chunk_28 저장 완료 (2900000번째 줄)
chunk_29 저장 완료 (3000000번째 줄)
chunk_30 저장 완료 (3100000번째 줄)
chunk_31 저장 완료 (3200000번째 줄)
chunk_32 저장 완료 (3300000번째 줄)
chunk_33 저장 완료 (3400000번째 줄)
chunk_34 저장 완료 (3500000번째 줄)
chun

In [65]:
import pandas as pd
import glob

# 메타데이터 product_id 목록
df_meta = pd.read_csv(r'C:\workspace\finalproject\data\amazon_main_pname_cleaned.csv')
valid_ids = set(df_meta['product_id'].tolist())
print(f"필터링할 product_id 수: {len(valid_ids)}")

# 모든 파케트 파일 읽으면서 필터링
parquet_files = glob.glob(r'C:\workspace\finalproject\data\reviews_chunk_*.parquet')
print(f"총 파케트 파일 수: {len(parquet_files)}")

filtered_chunks = []
for i, file in enumerate(parquet_files):
    df_chunk = pd.read_parquet(file)
    filtered = df_chunk[df_chunk['product_id'].isin(valid_ids)]
    if len(filtered) > 0:
        filtered_chunks.append(filtered)
    if i % 20 == 0:
        print(f"{i}번째 파일 처리 중...")

df_reviews = pd.concat(filtered_chunks, ignore_index=True)
print(f"\n최종 추출된 리뷰 수: {len(df_reviews)}")
print(df_reviews.head(3))

필터링할 product_id 수: 211265
총 파케트 파일 수: 240
0번째 파일 처리 중...
20번째 파일 처리 중...
40번째 파일 처리 중...
60번째 파일 처리 중...
80번째 파일 처리 중...
100번째 파일 처리 중...
120번째 파일 처리 중...
140번째 파일 처리 중...
160번째 파일 처리 중...
180번째 파일 처리 중...
200번째 파일 처리 중...
220번째 파일 처리 중...

최종 추출된 리뷰 수: 2407936
   product_id  rating                                        review_text  \
0  B08G81QQ9L     5.0  Bought this for my granddaughter.  Her entire ...   
1  B00V6R3R3S     5.0  Beautiful palette- very pleased! I got one as ...   
2  B014PVXY5W     1.0  I returned this item because it had terrible c...   

       timestamp  verified_purchase  
0  1612052493701               True  
1  1452647102000               True  
2  1672787634561               True  


In [66]:
import re

# 1. 결측치 확인
print(df_reviews[['product_id', 'rating', 'review_text']].isnull().sum())

# 2. 3점 제거 및 레이블 생성
df_reviews = df_reviews[df_reviews['rating'] != 3.0]
df_reviews['label'] = df_reviews['rating'].apply(lambda x: 1 if x >= 4 else 0)

# 3. 텍스트 정제
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_reviews['clean_review'] = df_reviews['review_text'].apply(clean_text)

# 4. 빈 텍스트 제거
df_reviews = df_reviews[df_reviews['clean_review'].str.len() > 0]

# 5. 중복 제거
df_reviews = df_reviews.drop_duplicates(subset='review_text')

print(f"\n전처리 후 행 수: {len(df_reviews)}")
print(df_reviews['label'].value_counts())

product_id     0
rating         0
review_text    0
dtype: int64

전처리 후 행 수: 2012598
label
1    1629399
0     383199
Name: count, dtype: int64


In [67]:
# 메타데이터 로드
df_meta = pd.read_csv(r'C:\workspace\finalproject\data\amazon_main_pname_cleaned.csv')

# merge
df_final = df_reviews.merge(df_meta, on='product_id', how='left')

# 확인
print(f"merge 후 행 수: {len(df_final)}")
print(f"결측치:\n{df_final.isnull().sum()}")
print(df_final.head(3).to_string())

merge 후 행 수: 2012598
결측치:
product_id               0
rating                   0
review_text              0
timestamp                0
verified_purchase        0
label                    0
clean_review             0
main_category            0
sub_category         82578
average_rating           0
dtype: int64
   product_id  rating                                                                                                                                                                                                                                                                                                                                    review_text      timestamp  verified_purchase  label                                                                                                                                                                                                                                                                                                      

In [68]:
# sub_category 결측치가 어떤 main_category에서 나오는지 확인
print(df_final[df_final['sub_category'].isna()]['main_category'].value_counts())

main_category
makeup_Makeup Palettes    39486
makeup_Eyes                9384
skincare_Face              8389
makeup_Face                8374
skincare_eyecare           7992
skincare_lipcare           4471
makeup_Lips                2965
suncare                    1517
Name: count, dtype: int64


In [69]:
# sub_category 결측치를 main_category로 채우기
df_final['sub_category'] = df_final['sub_category'].fillna(df_final['main_category'])

# 확인
print(f"결측치:\n{df_final.isnull().sum()}")
print(df_final['sub_category'].value_counts())

결측치:
product_id           0
rating               0
review_text          0
timestamp            0
verified_purchase    0
label                0
clean_review         0
main_category        0
sub_category         0
average_rating       0
dtype: int64
sub_category
Creams & Moisturizers          296713
Cleansers                      184018
Treatments & Masks             182926
Masks                          134269
Sunscreens                     125401
Balms & Moisturizers            97769
Mascara                         97658
Lipstick                        75409
Eyeliner                        69359
Eyeshadow                       65122
Self-Tanners                    60182
Lash Enhancers & Primers        55592
Eyebrow Color                   54906
Foundation                      50561
Lip Glosses                     41423
makeup_Makeup Palettes          39486
Creams                          39439
Toners & Astringents            39191
Foundation Primers              38195
Powder           

In [70]:
df_final.to_csv(r'C:\workspace\finalproject\data\beauty_care_reviews_final.csv', index=False)
print("저장 완료!")
print(f"최종 행 수: {len(df_final)}")
print(df_final['main_category'].value_counts())

저장 완료!
최종 행 수: 2012598
main_category
skincare_Face             734971
makeup_Eyes               358707
suncare                   223303
makeup_Face               202470
makeup_Lips               141144
maskpack                  114989
skincare_lipcare          110171
skincare_eyecare           87357
makeup_Makeup Palettes     39486
Name: count, dtype: int64


In [71]:
# verified_purchase True만 남기기
df_final = df_final[df_final['verified_purchase'] == True]

print(f"필터링 후 행 수: {len(df_final)}")
print(df_final['label'].value_counts())

필터링 후 행 수: 1743436
label
1    1403893
0     339543
Name: count, dtype: int64


In [72]:
# 컬럼명 변경
df_final = df_final.rename(columns={
    'rating': 'review_rating',          # 소비자가 남긴 별점
    'average_rating': 'product_avg_rating'  # 상품 평균 별점
})

# 불필요한 컬럼 제거 후 저장
df_final = df_final.drop(columns=['timestamp', 'verified_purchase'])

df_final.to_csv(r'C:\workspace\finalproject\data\beauty_care_reviews_final.csv', index=False)
print("저장 완료!")
print(df_final.columns.tolist())

저장 완료!
['product_id', 'review_rating', 'review_text', 'label', 'clean_review', 'main_category', 'sub_category', 'product_avg_rating']
